In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from plottable import ColumnDefinition, Table
from scipy import stats
import matplotlib.colors as mcolors


def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

In [6]:
path = "D:/bioTest/r/episcore/"

data_res = pd.read_csv(f"{path}/out_Les.csv")
data_res['Age Acceleration'] = data_res['Epigenetic Age (Bernabeu)'] - data_res['True Age']
    
xy_min, xy_max = np.quantile(data_res[['True Age', 'Epigenetic Age (Bernabeu)']].values.flatten(), [0.01, 0.99])
xy_ptp = xy_max - xy_min

n_rows = 2
n_cols = 2
fig_height = 5
fig_width = 7
sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), height_ratios=[2, 8],  width_ratios=[4, 2], gridspec_kw={'wspace':0.10, 'hspace': 0.05}, layout='constrained')

ds_table = pd.DataFrame(index=['MAE', r"Pearson $\rho$", 'Bias'], columns=[f'{data_res.shape[0]} FMBA samples'])
mae = mean_absolute_error(data_res['True Age'].values, data_res['Epigenetic Age (Bernabeu)'].values)
rho, _ = stats.pearsonr(data_res['True Age'].values, data_res['Epigenetic Age (Bernabeu)'].values)
bias = np.mean(data_res['Epigenetic Age (Bernabeu)'] - data_res['True Age'])
ds_table.at['MAE', f'{data_res.shape[0]} FMBA samples'] = f"{mae:0.3f}"
ds_table.at[ r"Pearson $\rho$", f'{data_res.shape[0]} FMBA samples'] = f"{rho:0.3f}"
ds_table.at['Bias', f'{data_res.shape[0]} FMBA samples'] = f"{bias:0.3f}"

col_defs = [
    ColumnDefinition(
        name="index",
        title='Metrics',
        textprops={"ha": "left"},
        width=4.5,
    ),
    ColumnDefinition(
        name=f'{data_res.shape[0]} FMBA samples',
        textprops={"ha": "center"},
        width=2.0,
    ),
]
table = Table(
    ds_table,
    column_definitions=col_defs,
    row_dividers=True,
    footer_divider=False,
    ax=axs[0, 0],
    textprops={"fontsize": 8},
    row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
    col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
    column_border_kw={"linewidth": 1, "linestyle": "-"},
).autoset_fontcolors(colnames=[f'{data_res.shape[0]} FMBA samples'])

scatter = sns.scatterplot(
    data=data_res,
    x='True Age',
    y="Epigenetic Age (Bernabeu)",
    linewidth=0.5,
    alpha=0.8,
    edgecolor="k",
    s=25,
    color='crimson',
    ax=axs[1, 0],
)
bisect = sns.lineplot(
    x=[xy_min - 0.15 * xy_ptp, xy_max + 0.15 * xy_ptp],
    y=[xy_min - 0.15 * xy_ptp, xy_max + 0.15 * xy_ptp],
    linestyle='--',
    color='black',
    linewidth=1.0,
    ax=axs[1, 0]
)
regplot = sns.regplot(
    data=data_res,
    x='True Age',
    y="Epigenetic Age (Bernabeu)",
    color='red',
    scatter=False,
    truncate=False,
    ax=axs[1, 0]
)
axs[1, 0].set_xlim(xy_min - 0.15 * xy_ptp, xy_max + 0.15 * xy_ptp)
axs[1, 0].set_ylim(xy_min - 0.15 * xy_ptp, xy_max + 0.15 * xy_ptp)
axs[1, 0].set_ylabel("Epigenetic Age (Bernabeu)")
axs[1, 0].set_xlabel("Age")

axs[0, 1].axis('off')

violin = sns.violinplot(
    data=data_res,
    x=[0] * data_res.shape[0],
    y='Age Acceleration',
    color=make_rgb_transparent(mcolors.to_rgb('crimson'), (1, 1, 1), 0.5),
    density_norm='width',
    saturation=0.75,
    linewidth=1.0,
    ax=axs[1, 1],
    legend=False,
)
axs[1, 1].set_ylabel('Age Acceleration')
axs[1, 1].set_xlabel('')
axs[1, 1].set(xticklabels=[]) 
axs[1, 1].set(xticks=[]) 
fig.savefig(f"{path}/Bernabeu.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path}/Bernabeu.pdf", bbox_inches='tight')
plt.close(fig)